# OPTUNA HYPERPARAMETER TUNING

### Imports

In [ ]:
#sklearn
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, cohen_kappa_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Metrics
from sklearn.metrics import f1_score, make_scorer

# Imbalanced-learn (SMOTE, Oversampling)
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, RandomOverSampler
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from sklearn.pipeline import make_pipeline

# Boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Optuna
import optuna

# Preprocessing
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessorLOEEALL

import numpy as np
import pandas as pd

#Warnings lightGBM
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)


### Dataset loading

In [ ]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

categorical_cols = ['Type', 'Gender', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'RescuerID', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength','Breed1', 'Breed2', 'State']

(10045, 32)
(10045,)
(4948, 32)
(4948,)


### Initialization

In [12]:
# Initializing the folds that we will be using
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Initializng the second evaluation technique
qwk = make_scorer(cohen_kappa_score, weights="quadratic")
scoring = {
        "f1_macro": "f1_macro",
        "QWK": qwk
    }

### XGBoost hyperparameter tuning

In [4]:
def objective_xgb(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.3)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)
    
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "mlogloss"    # for multiple classification
    }

    #Creamos pipeline con preprocessor y SMOTE
    pipe = imb_make_pipeline(
        preprocessorLOEEALL,
        SMOTE(random_state=42),      
        XGBClassifier(**params)
    )

    #Cross-validation estratificada
    score = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

    return np.mean(score['test_QWK'])

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
xgb_study = optuna.create_study(
    direction="maximize",
    study_name="xgb_study",
    storage="sqlite:///optuna_studies/xgb_study.db",  # se guarda en este archivo
    load_if_exists=True
)

xgb_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "min_child_weight": 1,
    "subsample": 1.0,
    "colsample_bytree": 1.0
})

xgb_study.optimize(objective_xgb, n_trials=100)

print(f"Mejores parámetros: {xgb_study.best_params}")

[I 2026-03-12 23:22:37,746] A new study created in RDB with name: xgb_study
[I 2026-03-12 23:22:47,576] Trial 0 finished with value: 0.37137433093726996 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 1}. Best is trial 0 with value: 0.37137433093726996.
[I 2026-03-12 23:24:00,818] Trial 1 finished with value: 0.31728641889443904 and parameters: {'looe_sigma': 0.18749658624250753, 'n_estimators': 703, 'max_depth': 9, 'learning_rate': 0.011275024590980353, 'subsample': 0.8393564241810125, 'colsample_bytree': 0.8620025959784515, 'min_child_weight': 3}. Best is trial 0 with value: 0.37137433093726996.
[I 2026-03-12 23:24:10,877] Trial 2 finished with value: 0.2582667863155529 and parameters: {'looe_sigma': 0.2863595359646975, 'n_estimators': 147, 'max_depth': 7, 'learning_rate': 0.07997429828663842, 'subsample': 0.7189335162802492, 'colsample_bytree': 0.9551676243460944, 'min_chil

Mejores parámetros: {'looe_sigma': 0.019652314650054948, 'n_estimators': 608, 'max_depth': 9, 'learning_rate': 0.023990406876147237, 'subsample': 0.6353477659511628, 'colsample_bytree': 0.6298885399507039, 'min_child_weight': 6}


### CatBoost hyperparameter tuning

In [ ]:
def objective_catboost(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.1)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)
    
    #Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log = True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20),
        "random_strength": 1,
        "bagging_temperature": 1,
        "border_count": 128,
        "verbose":0, 
        "allow_writing_files":False,
        "auto_class_weights" : "Balanced",
        "random_state": 42,
        "thread_count": -1
        
    }

    #Creamos pipeline con preprocessor y SMOTE
    pipe = make_pipeline(
        preprocessorLOEEALL,     
        CatBoostClassifier(**params)
    )

    #Cross-validation estratificada
    score = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

    return np.mean(score['test_QWK'])

# 4. Lanzamos la optimización
# (Asumiendo que ya tienes X_train e y_train)
cat_study = optuna.create_study(
    direction="maximize",
    study_name="cat_study_native",
    storage="sqlite:///optuna_studies/cat_study.db",  # se guarda en este archivo
    load_if_exists=True
)

cat_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 1000,         
    "max_depth": 6,               
    "learning_rate": 0.03,        
    "l2_leaf_reg": 3.0,           
    "random_strength": 1.0,       
    "bagging_temperature": 1.0    
})

cat_study.optimize(objective_catboost, n_trials=77)

print(f"Mejores parámetros: {cat_study.best_params}")

[I 2026-03-13 10:53:34,465] Using an existing study with name 'cat_study_native' instead of creating a new one.
[I 2026-03-13 10:54:15,390] Trial 28 finished with value: 0.405229867541044 and parameters: {'looe_sigma': 0.05, 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0}. Best is trial 18 with value: 0.4107981631250629.
[I 2026-03-13 10:54:40,700] Trial 29 finished with value: 0.40250729891701464 and parameters: {'looe_sigma': 0.017229564139207832, 'n_estimators': 620, 'max_depth': 6, 'learning_rate': 0.11630740297694062, 'l2_leaf_reg': 16.8591356829125}. Best is trial 18 with value: 0.4107981631250629.
[I 2026-03-13 10:55:12,993] Trial 30 finished with value: 0.39556003224148917 and parameters: {'looe_sigma': 0.01422533318362771, 'n_estimators': 778, 'max_depth': 6, 'learning_rate': 0.020369122957624965, 'l2_leaf_reg': 13.455675949506874}. Best is trial 18 with value: 0.4107981631250629.
[I 2026-03-13 10:56:26,803] Trial 31 finished with value: 0.4060

### RandomForest hyperparameter tuning

In [5]:
def objective_rf(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.3)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    # Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 700),
        "max_depth": trial.suggest_categorical("max_depth", [60,70,80,90,100,None]),
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "bootstrap": True,
        "random_state": 42,
        "n_jobs": -1
    }

    # Creamos pipeline con preprocessor y SMOTE
    pipe = imb_make_pipeline(
        preprocessorLOEEALL,
        RandomOverSampler(random_state=42),
        RandomForestClassifier(**params)
    )

    # Cross-validation estratificada
    score = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

    return np.mean(score['test_QWK'])

# Crear estudio de Optuna
rf_study = optuna.create_study(
    direction="maximize",
    study_name="rf_study",
    storage="sqlite:///optuna_studies/rf_study.db",
    load_if_exists=True
)

rf_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt"
})

# Lanzar optimización
rf_study.optimize(objective_rf, n_trials=60)

# Resultados
print(f"Mejores parámetros: {rf_study.best_params}")

[I 2026-03-13 00:01:28,733] A new study created in RDB with name: rf_study
[I 2026-03-13 00:01:32,775] Trial 0 finished with value: 0.3889739409262212 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.3889739409262212.
[I 2026-03-13 00:01:45,825] Trial 1 finished with value: 0.2999162070114205 and parameters: {'looe_sigma': 0.26815416360041905, 'n_estimators': 435, 'max_depth': None, 'max_features': 'log2'}. Best is trial 0 with value: 0.3889739409262212.
[I 2026-03-13 00:02:03,182] Trial 2 finished with value: 0.4072918244892015 and parameters: {'looe_sigma': 0.02389021008307124, 'n_estimators': 616, 'max_depth': 100, 'max_features': 'log2'}. Best is trial 2 with value: 0.4072918244892015.
[I 2026-03-13 00:02:10,712] Trial 3 finished with value: 0.3998156243047676 and parameters: {'looe_sigma': 0.03245282266736047, 'n_estimators': 239, 'max_depth': None, 'max_features': 'sqrt'}. Best is trial 2 with valu

Mejores parámetros: {'looe_sigma': 0.011222050950286213, 'n_estimators': 402, 'max_depth': 100, 'max_features': 'sqrt'}


### LightGBM hyperparameter tuning

In [6]:
def objective_lgbm(trial):
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.3)

    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    # Sugerimos hiperparámetros
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 700),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7 , 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1
    }

    # Creamos pipeline con preprocessor y SMOTE
    pipe = imb_make_pipeline(
        preprocessorLOEEALL,
        BorderlineSMOTE(random_state=42),
        LGBMClassifier(**params)
    )

    # Cross-validation estratificada
    score = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

    return np.mean(score['test_QWK'])


# Crear estudio de Optuna
lgbm_study = optuna.create_study(
    direction="maximize",
    study_name="lgbm_study",
    storage="sqlite:///optuna_studies/lgbm_study.db",
    load_if_exists=True
)

lgbm_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,          
    "max_depth": 15,              
    "num_leaves": 31,             
    "learning_rate": 0.1,         
    "min_child_samples": 20,      
    "subsample": 1.0,             
    "colsample_bytree": 1.0       
})

# Lanzar optimización
lgbm_study.optimize(objective_lgbm, n_trials=100)

# Resultados
print(f"Mejores parámetros: {lgbm_study.best_params}")

[I 2026-03-13 00:15:28,500] A new study created in RDB with name: lgbm_study
[I 2026-03-13 00:15:33,706] Trial 0 finished with value: 0.3870187050967445 and parameters: {'looe_sigma': 0.05, 'n_estimators': 100, 'max_depth': 15, 'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 20, 'subsample': 1.0, 'colsample_bytree': 1.0}. Best is trial 0 with value: 0.3870187050967445.
[I 2026-03-13 00:15:39,122] Trial 1 finished with value: 0.3253166817333433 and parameters: {'looe_sigma': 0.14439609245353652, 'n_estimators': 152, 'max_depth': 5, 'learning_rate': 0.099192980284241, 'num_leaves': 41, 'min_child_samples': 8, 'subsample': 0.9858617275929195, 'colsample_bytree': 0.7853878343638699}. Best is trial 0 with value: 0.3870187050967445.
[I 2026-03-13 00:15:45,524] Trial 2 finished with value: 0.2970853372966842 and parameters: {'looe_sigma': 0.1547714061747468, 'n_estimators': 299, 'max_depth': 4, 'learning_rate': 0.03517874885114156, 'num_leaves': 62, 'min_child_samples': 28, 'subs

Mejores parámetros: {'looe_sigma': 0.016156776522723557, 'n_estimators': 136, 'max_depth': 10, 'learning_rate': 0.05857075070621536, 'num_leaves': 36, 'min_child_samples': 14, 'subsample': 0.7943471901419035, 'colsample_bytree': 0.8876084957262791}


### Gradient Boosting hyperparameter tuning

In [ ]:
def objective_gbm(trial):
    # 1. Rango ajustado para tu target multiclase (0 a 4)
    looe_sigma = trial.suggest_float("looe_sigma", 0.0, 0.3)
    preprocessorLOEEALL.set_params(LOOE__sigma=looe_sigma)

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 6), 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
        "random_state": 42
    }

    # 3. Pipeline normal (make_pipeline) SIN técnicas de balanceo
    pipe = make_pipeline(
        preprocessorLOEEALL,
        GradientBoostingClassifier(**params)
    )

    # 4. Cross-validation estratificada
    score = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring)

    return np.mean(score['test_QWK'])


# Crear estudio de Optuna
gbm_study = optuna.create_study(
    direction="maximize",
    study_name="gbm_study",
    storage="sqlite:///optuna_studies/gbm_study.db",
    load_if_exists=True
)

# Valores por defecto reales de GradientBoostingClassifier
gbm_study.enqueue_trial({
    "looe_sigma": 0.05,
    "n_estimators": 100,
    "max_depth": 3,
    "learning_rate": 0.1,
    "min_samples_leaf": 1,
    "subsample": 1.0,
    "max_features": 1.0 
})

# Lanzar optimización
gbm_study.optimize(objective_gbm, n_trials=100)

# Resultados
print(f"Mejores parámetros: {gbm_study.best_params}")

### Voting and stacking classifiers

In [ ]:
"""
rf_best = RandomForestClassifier(**rf_study.best_params, random_state=42, n_jobs=-1)
lgbm_best = LGBMClassifier(**lgbm_study.best_params, random_state=42, n_jobs=-1, verbose=-1)
cat_best = CatBoostClassifier(**cat_study.best_params, random_state=42, verbose=0)
xgb_best = XGBClassifier(**xgb_study.best_params, random_state=42, n_jobs=-1)
"""

In [ ]:
rf_best = RandomForestClassifier(n_estimators =  573, max_depth = None, max_features = 'log2', random_state=42, n_jobs=-1)
lgbm_best = LGBMClassifier(n_estimators = 679, max_depth = 13, learning_rate = 0.021354407760849407, num_leaves = 110, min_child_samples = 13, subsample = 0.937569357477504, colsample_bytree = 0.6361451774580816, colsample_bytree = 0.6321534978115201, random_state=42, n_jobs=-1, verbose=-1)
cat_best = CatBoostClassifier(**cat_study.best_params, random_state=42, verbose=0)
xgb_best = XGBClassifier(**xgb_study.best_params, random_state=42, n_jobs=-1)

In [ ]:
pipe_rf = imb_make_pipeline(preprocessorLOEEALL, RandomOverSampler(random_state=42), rf_best)
pipe_lgbm = imb_make_pipeline(preprocessorLOEEALL, BorderlineSMOTE(random_state=42), lgbm_best)
pipe_cat = imb_make_pipeline(preprocessorLOEEALL, BorderlineSMOTE(random_state=42), cat_best)
pipe_xgb = imb_make_pipeline(preprocessorLOEEALL, RandomOverSampler(random_state=42), xgb_best)

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('cat', pipe_cat),
        ('xgb', pipe_xgb)
    ],
    voting='soft',   # TRUCO 1: 'soft' suele dar mejor métrica que 'hard'
    #weights=[1, 2, 2, 1] # TRUCO 2: Ponderación (opcional)
)

results_cv = cross_validate(voting_clf, X_train, y_train, cv=skf, scoring=scoring)

print(f"QWK del Ensemble: {np.mean(results_cv['test_QWK']):.4f}")

In [ ]:
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', pipe_rf),
        ('lgbm', pipe_lgbm),
        ('cat', pipe_cat),
        ('xgb', pipe_xgb)
    ],
    # TRUCO 1: El Meta-Learner debe ser un modelo simple
    final_estimator=LogisticRegression(max_iter=1000, random_state=42), 
    
    cv=skf, 
    
    # TRUCO 3: 'passthrough=False' (por defecto) para evitar overfitting
    passthrough=False,
    
    n_jobs=-1 # Para no colapsar la RAM
)